In [1]:
import pandas as pd
import numpy as np
from pandas.api.types import CategoricalDtype
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
from catboost import CatBoostRegressor, Pool

cols = [
    # Для идентификации
    "date",

    # таргет
    "units",

    # прошлое
    "units_yesterday", "units_prev_week",
    # "rolling_mean_4w", - кривая фича, переделать, она не должна включать день предсказания

    # категориальные
    "store_code", "store_item_code",

    # погода (float32)
    "tmax", "tmin", "tavg", "depart", "dewpoint", "wetbulb", "heat", "cool",
    "sunrise", "sunset",
    "snowfall", "preciptotal", "stnpressure", "sealevel",
    "resultspeed", "resultdir", "avgspeed",

    # календарь и флаги (int16)
    "year", "week", "BCFG", "BLDU", "BLSN", "BR", "DU", "DZ", "FG", "FU",
    "FZDZ", "FZFG", "FZRA", "GR", "GS", "HZ", "MIFG", "PL", "PRFG", "RA",
    "SG", "SN", "SQ", "TS", "TSRA", "TSSN", "UP", "VCFG", "VCTS",
    "day_of_week", "month", "is_weekend", "is_holiday",
    "rain_streak", "dry_streak",

    # look‑ahead
    "avg_temp_next_day", "rain_next_day", "days_to_holiday"
]

dtypes = {
    # целевой
    "units": "int16",        # -32 768 … 32 767

    # Для идентификации
    "date": "object",

    # прошлое → float32
    **{c: "float32" for c in [
        "units_yesterday", "units_prev_week", "rolling_mean_4w",
    ]},
    

    # категориальные коды
    "store_code": "category",
    "store_item_code": "category",

    # погода → float32
    **{c: "float32" for c in [
        "tmax","tmin","tavg","depart","dewpoint","wetbulb","heat","cool",
        "sunrise","sunset",
        "snowfall","preciptotal","stnpressure","sealevel",
        "resultspeed","resultdir","avgspeed",
        "avg_temp_next_day","rain_next_day",
    ]},

    # календарные/флаговые → int16
    **{c: "int16" for c in [
        "year","week","day_of_week","month",
        "is_weekend","is_holiday","rain_streak","dry_streak",
        "BCFG","BLDU","BLSN","BR","DU","DZ","FG","FU","FZDZ","FZFG",
        "FZRA","GR","GS","HZ","MIFG","PL","PRFG","RA","SG","SN","SQ",
        "TS","TSRA","TSSN","UP","VCFG","VCTS", "days_to_holiday",
    ]},
}

full_table_df = pd.read_csv(
    "./data/big_full_table.csv",
    usecols=cols,
    dtype=dtypes,
)
full_table_df['date'] = pd.to_datetime(full_table_df['date'])

In [2]:
from catboost import CatBoostRegressor

model = CatBoostRegressor()
model.load_model(f"../ml-models/CatBoost v1.cbm")


In [3]:
full_table_df[model.feature_names_][full_table_df[model.feature_names_].isna().any(axis=1)]

,store_code,store_item_code,units_yesterday,units_prev_week,tmax,tmin,tavg,depart,dewpoint,wetbulb,...,VCTS,day_of_week,month,is_weekend,is_holiday,rain_streak,dry_streak,avg_temp_next_day,rain_next_day,days_to_holiday


In [4]:
cat_cols = ["store_code", "store_item_code"]
test_pool   = Pool(full_table_df[model.feature_names_],  full_table_df['units'],  cat_features=cat_cols)

# Предсказываем.
Y_production = model.predict(test_pool)

In [6]:
df = pd.DataFrame({'y_pred': Y_production, 'units': full_table_df['units'], 'date': full_table_df['date'], 'store_item_code': full_table_df['store_item_code']})
df.loc[df['y_pred'] < 0, 'y_pred'] = 0

# Выводим случайные 40 строк
display(df.sample(40))

,y_pred,units,date,store_item_code
125617,0.161990,0,2013-05-21,s37-i38
115386,0.203479,0,2013-04-03,s45-i50
215659,10.787509,8,2014-07-25,s19-i83
50483,88.120361,78,2012-07-21,s4-i9
3678,0.000000,0,2012-01-15,s5-i105
70024,67.631445,50,2012-10-06,s24-i6
192852,53.498205,116,2014-04-05,s24-i6
127434,12.357323,14,2013-05-31,s4-i27
95750,7.506490,10,2013-01-16,s19-i83
230924,0.444547,0,2014-10-07,s35-i63


In [14]:
full_table_df['units_pred'] = Y_production
full_table_df.to_csv('./data/prediction_real_weather-0.1.csv', index=False)